In [1]:
import time
import heapq

# turn this off if your terminal no show colours
USE_COLOUR = True

# colours for the maze display
BLUE = "\033[94m"
RED = "\033[91m"
GREEN = "\033[92m"
YELLOW = "\033[93m"
RESET = "\033[0m"

# 0 means open cell, 1 means wall
maze = [
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [1, 1, 0, 0, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 0, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
    [1, 1, 1, 0, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
]


# Helper function: controls coloured terminal output only; it does not affect the GBFS search.
def colour_text(text, colour):
    # add colour code before and after the text
    if USE_COLOUR:
        return colour + text + RESET
    return text


# Checks whether a position can be used as a state in the search.
# A valid state must be inside the grid and must contain 0 (not a wall).
def is_valid_open_cell(position, maze_data):
    row, col = position

    # check if the position is inside the maze and not a wall
    return (0 <= row < len(maze_data) and
            0 <= col < len(maze_data[0]) and
            maze_data[row][col] == 0)


# Generates all valid successor states from the current cell.
# The branching factor is at most 4 because movement is limited to four directions.
def get_neighbours(position, maze_data):
    row, col = position

    # four possible movements: up, down, left and right
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    neighbours = []

    for row_change, col_change in directions:
        neighbour = (row + row_change, col + col_change)

        # only add cells that can be moved to
        if is_valid_open_cell(neighbour, maze_data):
            neighbours.append(neighbour)

    return neighbours


# Heuristic used by GBFS: h(n) = Manhattan distance.
# It estimates closeness to the goal but does not consider walls or the path cost g(n).
def heuristic(position, goal):
    # Manhattan distance estimates how far the goal is
    return abs(position[0] - goal[0]) + abs(position[1] - goal[1])


# Reconstructs the final solution after the goal is found.
# parent[child] stores the state that generated that child.
def reconstruct_path(parent, start, goal):
    path = []
    current = goal

    # go backwards from goal until reaching the start
    while current != start:
        path.append(current)
        current = parent[current]

    path.append(start)

    # reverse it to get start -> goal
    path.reverse()
    return path


# Converts the coordinate path into readable directions for the program output.
def get_movement_sequence(path):
    movements = []

    # compare each cell with the next cell
    for index in range(len(path) - 1):
        row, col = path[index]
        next_row, next_col = path[index + 1]

        if next_row == row - 1:
            movements.append("Up")
        elif next_row == row + 1:
            movements.append("Down")
        elif next_col == col - 1:
            movements.append("Left")
        else:
            movements.append("Right")

    return movements


# Main Greedy Best-First Search function.
# Main idea: always expand the frontier node with the smallest heuristic value h(n).
# GBFS is greedy because node selection uses h(n) only and ignores accumulated path cost g(n).
def gbfs(maze_data, start, goal):
    # record the starting time to calculate execution time
    start_time = time.perf_counter()

    # priority queue for GBFS
    frontier = []

    # used to break ties when heuristic values are the same
    counter = 0

    # GBFS chooses the node with the smallest heuristic value
    heapq.heappush(frontier, (heuristic(start, goal), counter, start))

    # keep track of nodes already added to the search
    visited = {start}

    # store where each node came from
    parent = {}

    # statistics for the search
    nodes_generated = 1
    nodes_expanded = 0
    max_frontier_size = 1

    # save frontier information for display later
    frontier_history = []

    # save the order in which nodes are expanded
    expanded_nodes = []

    # Continue searching while there are still candidate states in the frontier.
    while frontier:
        # remove the node with the smallest heuristic value
        # Heap entries are (heuristic, counter, position).
        # The first two values are not needed after removal, so '_' is used to ignore them.
        _, _, current = heapq.heappop(frontier)

        # A node is counted as expanded when it is removed from the frontier for processing.
        nodes_expanded += 1
        expanded_nodes.append(current)

        # stop when the goal is reached
        if current == goal:
            path = reconstruct_path(parent, start, goal)
            return make_result(True, path, nodes_generated, nodes_expanded,
                               max_frontier_size, start_time, frontier_history,
                               expanded_nodes)

        # check all possible next cells
        for neighbour in get_neighbours(current, maze_data):
            # only add a node if it has not been visited
            if neighbour not in visited:
                # Mark the neighbour immediately when it enters the frontier.
                # This prevents the same cell from being generated and added more than once.
                visited.add(neighbour)

                # remember the previous node for path reconstruction
                parent[neighbour] = current

                # If two nodes have the same h(n), counter acts as a tie-breaker in the heap.
                counter += 1

                # add neighbour using its heuristic value
                heapq.heappush(frontier, (heuristic(neighbour, goal), counter, neighbour))

                nodes_generated += 1

        # update the largest frontier size
        max_frontier_size = max(max_frontier_size, len(frontier))

        # save current frontier for the table
        frontier_history.append({
            "expanded": current,
            "frontier": [item[2] for item in frontier]
        })

    # no path was found
    return make_result(False, [], nodes_generated, nodes_expanded,
                       max_frontier_size, start_time, frontier_history,
                       expanded_nodes)


# Packages the search outcome and performance measurements into one dictionary.
def make_result(success, path, generated, expanded, max_frontier, start_time,
                frontier_history, expanded_nodes):

    # put all search results into one dictionary
    return {
        "success": success,
        "path": path,
        "movement_sequence": get_movement_sequence(path),
        # With movement cost = 1, path cost equals the number of moves,
        # which is one less than the number of coordinates in the path.
        "solution_cost": len(path) - 1 if success else 0,
        "nodes_generated": generated,
        "nodes_expanded": expanded,
        "max_frontier_size": max_frontier,

        # calculate total running time
        "execution_time": time.perf_counter() - start_time,

        "frontier_history": frontier_history,
        "expanded_nodes": expanded_nodes
    }


# Chooses the symbol/colour for one maze cell.
# This function is only for visualisation and does not influence the search.
def get_cell_display(position, maze_data, path_set, expanded_set, start, goal):
    row, col = position

    # decide what symbol to show for each cell
    if position == start:
        return colour_text("S ", GREEN)
    elif position == goal:
        return colour_text("G ", YELLOW)
    elif position in path_set:
        return colour_text("* ", RED)
    elif position in expanded_set:
        return colour_text("* ", BLUE) if USE_COLOUR else "+ "
    elif maze_data[row][col] == 1:
        return "# "

    return ". "


# Displays walls, unvisited cells, expanded cells, final path, start and goal.
# This is presentation/output logic, not part of the GBFS decision process.
def display_maze(maze_data, start, goal, path, expanded_nodes):
    # convert lists into sets for faster checking
    path_set = set(path)
    expanded_set = set(expanded_nodes)

    print("+" + "--" * len(maze_data[0]) + "+")

    for row in range(len(maze_data)):
        line = "|"

        for col in range(len(maze_data[0])):
            line += get_cell_display((row, col), maze_data, path_set,
                                     expanded_set, start, goal)

        print(line + "|")

    print("+" + "--" * len(maze_data[0]) + "+")

    if USE_COLOUR:
        print("Legend: # Wall | . Unvisited | " + colour_text("* Expanded", BLUE) + " | " + colour_text("* Final path", RED) + " | S Start | G Goal")
    else:
        print("Legend: # Wall | . Unvisited | + Expanded | * Final path | S Start | G Goal")


# Prints the recorded frontier after each expansion so the search process can be explained step by step.
def display_frontier_history(history):
    print("\n" + "=" * 75)
    print("FRONTIER EXPANSION TABLE: GREEDY BEST-FIRST SEARCH (GBFS)")
    print("=" * 75)

    # format the table into three columns
    print("{:<8} {:<20} {:<40}".format("Step", "Expanded Node", "Frontier"))
    print("-" * 75)

    for step, record in enumerate(history, 1):
        # convert frontier nodes into one string
        frontier = ", ".join(str(node) for node in record["frontier"]) or "Empty"

        print("{:<8} {:<20} {:<40}".format(
            step, str(record["expanded"]), frontier))

    print("-" * 75)


# Displays the detailed result for one test case, including path and performance measurements.
def display_result(test_number, start, goal, result):
    print("=" * 72)
    print("TEST CASE {}".format(test_number))
    print("=" * 72)

    print("Start position: {}".format(start))
    print("\nGoal position : {}".format(goal))

    if result["success"]:
        print("\nMovement sequence:")

        # display every movement step
        for step, direction in enumerate(result["movement_sequence"], start=1):
            print("Step {:>2}: {} -> {:<5} -> {}".format(
                step, result["path"][step - 1], direction, result["path"][step]))

    print("\nStatus: {}".format(
        "Path found" if result["success"] else "Path not found"))

    print("\nReason: {}".format(
        "Goal reached" if result["success"] else "No path found"))

    print("\nPath length (cost): {} move(s)".format(result["solution_cost"]))
    print("\nNodes generated: {}".format(result["nodes_generated"]))
    print("\nNodes expanded: {}".format(result["nodes_expanded"]))
    print("\nMaximum frontier size: {}".format(result["max_frontier_size"]))
    print("\nExecution time: {:.8f} seconds".format(result["execution_time"]))

    display_frontier_history(result["frontier_history"])

    if result["success"]:
        print("\nSolution path:")
        print(result["path"])

    print("\nMaze Visualisation:")
    display_maze(maze, start, goal, result["path"], result["expanded_nodes"])


# Combines all 10 test-case results into one comparison summary.
def display_summary(results):
    print("\n" + "=" * 104)
    print("SUMMARY TABLE: GREEDY BEST-FIRST SEARCH (GBFS)")
    print("=" * 104)

    # print the headings with fixed spacing
    print(f"{'Test':<8}{'Success':<12}{'Path Length':<16}{'Nodes Generated':<18}{'Nodes Expanded':<18}{'Maximum Frontier':<22}{'Time (seconds)'}")
    print("-" * 104)

    # enumerate(..., 1) numbers the test cases starting from 1 instead of 0.
    for number, result in enumerate(results, 1):
        print(f"{number:<8}{str(result['success']):<12}{result['solution_cost']:<16}{result['nodes_generated']:<18}{result['nodes_expanded']:<18}{result['max_frontier_size']:<22}{result['execution_time']:.8f}")

    # keep only the test cases that found a path
    successful = [result for result in results if result["success"]]

    # Total number of test cases, used when calculating averages and success rate.
    count = len(results)

    print("\n" + "=" * 104)
    print("OVERALL RESULT: GREEDY BEST-FIRST SEARCH (GBFS)")
    print("=" * 104)

    print(f"Success Rate: {len(successful) / count * 100:.2f}%")

    # calculate average cost only from successful searches
    print(f"Average Solution Cost: {sum(item['solution_cost'] for item in successful) / len(successful) if successful else 0:.2f}")

    print(f"Average Nodes Generated: {sum(item['nodes_generated'] for item in results) / count:.2f}")
    print(f"Average Nodes Expanded: {sum(item['nodes_expanded'] for item in results) / count:.2f}")
    print(f"Average Maximum Frontier Size: {sum(item['max_frontier_size'] for item in results) / count:.2f}")
    print(f"Average Execution Time: {sum(item['execution_time'] for item in results) / count:.8f} seconds")


# Same 10 test cases are used so comparison is fair.
test_cases = [
    {"start": (0, 0), "goal": (9, 9)}, {"start": (0, 2), "goal": (8, 9)},
    {"start": (2, 0), "goal": (9, 9)}, {"start": (4, 0), "goal": (9, 9)},
    {"start": (6, 0), "goal": (9, 9)}, {"start": (8, 0), "goal": (0, 9)},
    {"start": (8, 2), "goal": (0, 9)}, {"start": (4, 3), "goal": (0, 9)},
    {"start": (6, 5), "goal": (8, 9)}, {"start": (0, 9), "goal": (8, 0)}
]

# Stores the result dictionary returned by each test case for the final summary.
all_results = []

print("=" * 72)
print("MAZE SOLVING USING GREEDY BEST-FIRST SEARCH (GBFS)")
print("=" * 72)

print("Maze size: {} x {}".format(len(maze), len(maze[0])))
print("Allowed movement: Up, Down, Left, Right")
print("Cost per movement: 1")
print("Heuristic: Manhattan distance")
print()

# Run exactly the same GBFS procedure for each of the 10 predefined test cases.
for test_number, case in enumerate(test_cases, 1):
    start = case["start"]
    goal = case["goal"]

    # run GBFS for the current test case
    result = gbfs(maze, start, goal)

    # save the result for the final summary
    all_results.append(result)

    # Show this test case immediately before moving to the next one.
    display_result(test_number, start, goal, result)

# show the results of all test cases
display_summary(all_results)

MAZE SOLVING USING GREEDY BEST-FIRST SEARCH (GBFS)
Maze size: 10 x 10
Allowed movement: Up, Down, Left, Right
Cost per movement: 1
Heuristic: Manhattan distance

TEST CASE 1
Start position: (0, 0)

Goal position : (9, 9)

Movement sequence:
Step  1: (0, 0) -> Right -> (0, 1)
Step  2: (0, 1) -> Right -> (0, 2)
Step  3: (0, 2) -> Down  -> (1, 2)
Step  4: (1, 2) -> Down  -> (2, 2)
Step  5: (2, 2) -> Left  -> (2, 1)
Step  6: (2, 1) -> Left  -> (2, 0)
Step  7: (2, 0) -> Down  -> (3, 0)
Step  8: (3, 0) -> Down  -> (4, 0)
Step  9: (4, 0) -> Down  -> (5, 0)
Step 10: (5, 0) -> Down  -> (6, 0)
Step 11: (6, 0) -> Right -> (6, 1)
Step 12: (6, 1) -> Right -> (6, 2)
Step 13: (6, 2) -> Right -> (6, 3)
Step 14: (6, 3) -> Down  -> (7, 3)
Step 15: (7, 3) -> Down  -> (8, 3)
Step 16: (8, 3) -> Right -> (8, 4)
Step 17: (8, 4) -> Right -> (8, 5)
Step 18: (8, 5) -> Right -> (8, 6)
Step 19: (8, 6) -> Right -> (8, 7)
Step 20: (8, 7) -> Right -> (8, 8)
Step 21: (8, 8) -> Right -> (8, 9)
Step 22: (8, 9) -> Down 